# NYC TLC BIE pipeline orchestrator

Runs the pipeline end to end:
1. Download TLC monthly Parquet + zone lookup, upload to S3 `raw/`.
2. Create Athena external tables + register partitions.
3. Run the CTAS ETL (dims + fact) into S3 `curated/`.
4. Run the data-quality gate. **Any failure stops the run.**
5. Create the Redshift schema and `COPY` the curated Parquet.

SQL lives in the repo (`sql/`, `dq/`, `redshift/`); this notebook only
orchestrates and enforces the gate. Fill `config.yaml` first.

In [ ]:
import time, io, pathlib, re
import boto3, requests, yaml

CFG = yaml.safe_load(open('../config.yaml'))
REGION = CFG['region']
BUCKET = CFG['s3']['bucket']
RAW = CFG['s3']['raw_prefix']
ATHENA_DB = CFG['athena']['database']
ATHENA_WG = CFG['athena']['workgroup']
ATHENA_OUT = f"s3://{BUCKET}/{CFG['s3']['athena_results_prefix']}"
MONTHS = CFG['data']['months']
SERVICE = CFG['data']['service']
TLC_BASE = 'https://d37ci6vzurychx.cloudfront.net'

s3 = boto3.client('s3', region_name=REGION)
athena = boto3.client('athena', region_name=REGION)
print('config loaded:', BUCKET, REGION, len(MONTHS), 'months')

## 1. Ingest: download TLC files, upload to S3 raw/ (partition-style paths)

In [ ]:
def upload_month(month: str):
    fname = f'{SERVICE}_tripdata_{month}.parquet'
    url = f'{TLC_BASE}/trip-data/{fname}'
    key = f'{RAW}{SERVICE}/trip_month={month}/{fname}'
    # skip if already uploaded
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        print('exists, skip:', key); return
    except s3.exceptions.ClientError:
        pass
    print('downloading', url)
    r = requests.get(url, timeout=300); r.raise_for_status()
    s3.put_object(Bucket=BUCKET, Key=key, Body=r.content)
    print('uploaded', key, f'({len(r.content)/1e6:.1f} MB)')

for m in MONTHS:
    upload_month(m)

# zone lookup
zl = requests.get(f'{TLC_BASE}/misc/taxi_zone_lookup.csv', timeout=60); zl.raise_for_status()
s3.put_object(Bucket=BUCKET, Key=f'{RAW}zone_lookup/taxi_zone_lookup.csv', Body=zl.content)
print('zone lookup uploaded')

## Athena helper: run a statement, wait, return status/result

In [ ]:
def athena_run(sql: str, wait=True):
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': ATHENA_DB},
        WorkGroup=ATHENA_WG,
        ResultConfiguration={'OutputLocation': ATHENA_OUT},
    )['QueryExecutionId']
    if not wait:
        return qid
    while True:
        st = athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status']['State']
        if st in ('SUCCEEDED','FAILED','CANCELLED'):
            break
        time.sleep(2)
    if st != 'SUCCEEDED':
        reason = athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status'].get('StateChangeReason','')
        raise RuntimeError(f'Athena query {st}: {reason}\n{sql[:200]}')
    return qid

def athena_rows(qid):
    res = athena.get_query_results(QueryExecutionId=qid)
    rows = res['ResultSet']['Rows']
    header = [c['VarCharValue'] for c in rows[0]['Data']]
    out = []
    for r in rows[1:]:
        out.append({header[i]: c.get('VarCharValue') for i, c in enumerate(r['Data'])})
    return out

def run_sql_file(path, bucket=BUCKET):
    """Run each ;-delimited statement in a .sql file (after <BUCKET> substitution)."""
    text = pathlib.Path(path).read_text().replace('<BUCKET>', bucket)
    # strip line comments, split on ';'
    stmts = [s.strip() for s in re.sub(r'--[^\n]*', '', text).split(';') if s.strip()]
    for s in stmts:
        athena_run(s)
    print(f'ran {len(stmts)} statements from {path}')

## 2-3. External tables, CBD zone ref, then CTAS ETL (dims + fact)

In [ ]:
# NOTE: drop any prior CTAS outputs before re-running (CTAS fails if the
# target already exists). Drop the tables and clear s3 curated/ if re-running.
run_sql_file('../sql/00_athena_raw_external.sql')
run_sql_file('../sql/04_cbd_zone_reference.sql')
run_sql_file('../sql/02_ctas_dimensions.sql')
run_sql_file('../sql/03_ctas_fact_trips.sql')
print('ETL complete: dims + fact materialized to s3 curated/')

## 4. Data-quality gate  —  ANY failure stops the pipeline

Runs each check in `dq/dq_checks.sql`; if any check's `failures` column is > 0,
raises and does **not** proceed to the Redshift load.

In [ ]:
dq_text = pathlib.Path('../dq/dq_checks.sql').read_text().replace('<BUCKET>', BUCKET)
checks = [s.strip() for s in re.sub(r'--[^\n]*', '', dq_text).split(';') if s.strip()]

failed = []
for c in checks:
    qid = athena_run(c)
    for row in athena_rows(qid):
        name = row.get('check_name', '?')
        f = int(row.get('failures', '0') or 0)
        flag = 'FAIL' if f > 0 else 'ok'
        print(f'[{flag}] {name}: failures={f}  {row}')
        if f > 0:
            failed.append((name, f, row))

if failed:
    raise RuntimeError(f'DATA-QUALITY GATE FAILED ({len(failed)} checks). '
                       f'Load aborted. Fix before COPY: {failed}')
print('\nDQ GATE PASSED — safe to load to Redshift.')

## 5. Redshift: create schema and COPY curated Parquet

Uses redshift-data API (no cluster credentials in the notebook). Fill the
Redshift settings in `config.yaml`. The COPY role must have `s3:GetObject`
on the bucket and be attached to the Redshift namespace.

In [ ]:
rs = boto3.client('redshift-data', region_name=REGION)
RS = CFG['redshift']

def rs_run(sql):
    r = rs.execute_statement(WorkgroupName=RS['workgroup'], Database=RS['database'], Sql=sql)
    sid = r['Id']
    while True:
        d = rs.describe_statement(Id=sid)
        if d['Status'] in ('FINISHED','FAILED','ABORTED'):
            break
        time.sleep(1.5)
    if d['Status'] != 'FINISHED':
        raise RuntimeError(f"Redshift {d['Status']}: {d.get('Error','')}\n{sql[:200]}")
    return sid

def rs_run_file(path):
    text = (pathlib.Path(path).read_text()
            .replace('<BUCKET>', BUCKET)
            .replace('<REDSHIFT_COPY_ROLE_ARN>', RS['copy_role_arn']))
    stmts = [s.strip() for s in re.sub(r'--[^\n]*', '', text).split(';') if s.strip()]
    for s in stmts:
        rs_run(s)
    print(f'ran {len(stmts)} statements from {path}')

rs_run_file('../redshift/01_ddl.sql')
rs_run_file('../redshift/02_copy.sql')
print('Redshift load complete. Point QuickSight at the Redshift workgroup next.')

## 6. QuickSight (manual, ~15 min) + TEARDOWN

**QuickSight:** New dataset → Redshift (auto-discovered in `us-east-1`) →
schema `tlc` → create a custom SQL dataset joining `fact_trips` to the dims, or
use direct query. Build: KPI tiles (trips, revenue, avg fare), a monthly trend
line split by CBD/non-CBD, a zone map (`location_id`), and an hour×daypart
heatmap. Screenshot for the portfolio.

**Teardown (do this to avoid a bill):**
```python
# Pause or delete the Redshift Serverless workgroup after screenshots.
# Cancel the QuickSight subscription (Manage QuickSight → Account settings).
# Optionally empty the S3 bucket:
#   aws s3 rm s3://<bucket>/ --recursive
```